In [ ]:
import os
import io
import json
import hashlib
import shutil
from pathlib import Path
from collections import defaultdict

In [ ]:
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

## MOUNT COLAB CONTENT

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pathlib

## KAGGLE API SETUP

In [ ]:
def configure_kaggle():
    from google.colab import userdata
    token = userdata.get("KAGGLE_API_TOKEN")
    kaggle_dir = pathlib.Path.home() / ".kaggle"
    kaggle_dir.mkdir(exist_ok=True)
    key_path = kaggle_dir / "kaggle.json"
    key_path.write_text(token)
    key_path.chmod(0o600)
    print("Kaggle credentials written to", key_path)

In [ ]:
## CONFIGURE KAGGLE API TOKEN SECRETS

In [ ]:
from google.colab import userdata
userdata.get('KAGGLE_API_TOKEN')

## CONECT TO COLAB SECRETS

In [ ]:
configure_kaggle()

Kaggle credentials written to /root/.kaggle/kaggle.json


##  DOWNLOAAD DIVERSE-TOOLS-IMAGE-DATASET-4-ML
https://www.kaggle.com/datasets/oortdatahub/diverse-tools-image-dataset-for-machine-learning

In [ ]:
# Download and extract the dataset

In [ ]:
DATASET_SLUG = "oortdatahub/diverse-tools-image-dataset-for-machine-learning"
DOWNLOAD_DIR = Path("/content/dataset_raw") # 42GB download
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
## DOWNLOADING DATASET FROM KAGGLE

In [ ]:
os.system(f"kaggle datasets download -d {DATASET_SLUG} -p {DOWNLOAD_DIR} --unzip")


0

In [ ]:
## SET UP LOCAL DIRECTORY

In [ ]:
DATASET_ROOT = DOWNLOAD_DIR  # adjust if the unzip created one extra nesting level

In [ ]:
category_dirs = sorted([d for d in DATASET_ROOT.iterdir() if d.is_dir()])

In [ ]:
print(f"Found {len(category_dirs)} top-level category folders")
for d in category_dirs[:10]:
    print(" -", d.name)

Found 89 top-level category folders
 - Air Compressors Dataset
 - Angle Grinder Dataset Collection
 - Axe AI Dataset Collection
 - Ball Valve Dataset
 - Bearing Dataset
 - Caulking Gun Dataset Collection
 - Chisel Dataset
 - Clamp Dataset Collection
 - Concrete Mixer Dataset Collection
 - Construction Earmuffs Dataset


Walk every category folder recursively, collecting every image file path
along with its category label and its immediate parent folder (source batch),
since images from the same timestamp batch are the most likely source of
near-duplicates or exact duplicates.

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


In [ ]:
records = []

for category_dir in tqdm(category_dirs, desc="Scanning categories"):

    class_name = category_dir.name

    for file_path in category_dir.rglob("*"):

        if file_path.is_file() and file_path.suffix.lower() in IMAGE_EXTENSIONS:

            records.append({

                "class_name": class_name,
                "source_batch": file_path.parent.name,
                "file_path": str(file_path),
                "file_size_bytes": file_path.stat().st_size,

            })


Scanning categories:   0%|          | 0/89 [00:00<?, ?it/s]

In [ ]:
##  Dataset -> pandas df

In [ ]:
df = pd.DataFrame(records)

In [ ]:
print(f"Total image files found: {len(df)}")
print(f"Classes found: {df['class_name'].nunique()}")


Total image files found: 67121
Classes found: 89


Class distribution: raw file counts per class before any cleaning.

Compare against the Kaggle listing to spot missing or extra files.


In [ ]:
class_counts = df["class_name"].value_counts().sort_values()
print(class_counts)

class_name
Toolbelt Dataset                  72
Paint Spray Gun Dataset          196
Digital Torque Wrench Dataset    270
Generator Dataset Collection     284
Float Tool Dataset Collection    309
                                ... 
Rubber Gloves Dataset            989
Metal Nut Dataset                990
Traffic Cone Dataset             992
Welding Gloves Dataset           993
Socket Wrench Dataset            996
Name: count, Length: 89, dtype: int64


### Compute SHA-256 hash and pixel resolution for every image.


In [ ]:

def audit_single_image(file_path: str) -> dict:
    """Read one image file, return its hash, resolution, and corruption status."""
    path = Path(file_path)
    result = {
        "sha256": None,
        "width": None,
        "height": None,
        "is_corrupt": False,
        "error": None,
    }

    try:
        raw_bytes = path.read_bytes()
        result["sha256"] = hashlib.sha256(raw_bytes).hexdigest()

        with Image.open(io.BytesIO(raw_bytes)) as img:
            img.verify()

        # verify() invalidates the image object, so reopen to read dimensions
        with Image.open(io.BytesIO(raw_bytes)) as img:
            result["width"], result["height"] = img.size

    except Exception as exc:
        result["is_corrupt"] = True
        result["error"] = str(exc)

    return result


In [ ]:
## Audit Reports

In [ ]:
audit_results = []
for file_path in tqdm(df["file_path"], desc="Hashing and verifying images"):
    audit_results.append(audit_single_image(file_path))


Hashing and verifying images:   0%|          | 0/67121 [00:00<?, ?it/s]

In [ ]:
audit_df = pd.DataFrame(audit_results)

In [ ]:

df = pd.concat([df.reset_index(drop=True), audit_df], axis=1)


In [ ]:
corrupt_df = df[df["is_corrupt"]]
print(f"Corrupt or unreadable files: {len(corrupt_df)}")

Corrupt or unreadable files: 2


In [ ]:
if len(corrupt_df) > 0:
    print(corrupt_df[["class_name", "file_path", "error"]].head(20))


                               class_name  \
8563   Digital Caliper Dataset Collection   
29024           Ladder Dataset Collection   

                                               file_path  \
8563   /content/dataset_raw/Digital Caliper Dataset C...   
29024  /content/dataset_raw/Ladder Dataset Collection...   

                                                   error  
8563   cannot identify image file <_io.BytesIO object...  
29024  cannot identify image file <_io.BytesIO object...  


## Duplicate detection via exact SHA-256 match.
Flag every row whose hash appears more than once,
and separately report
duplicates that occur within the same class vs. across different classes


In [ ]:
clean_df = df[~df["is_corrupt"]].copy()

In [ ]:
hash_counts = clean_df["sha256"].value_counts()
duplicate_hashes = hash_counts[hash_counts > 1].index


In [ ]:
clean_df["is_duplicate"] = clean_df["sha256"].isin(duplicate_hashes)


In [ ]:
dup_df = clean_df[clean_df["is_duplicate"]].sort_values("sha256")
print(f"Files involved in exact duplicate groups: {len(dup_df)}")

Files involved in exact duplicate groups: 4944


In [ ]:
cross_class_dupes = (
    dup_df.groupby("sha256")["class_name"]
    .nunique()
    .loc[lambda s: s > 1]
)

In [ ]:
print(f"Duplicate groups spanning more than one class: {len(cross_class_dupes)}")
if len(cross_class_dupes) > 0:
    print(dup_df[dup_df["sha256"].isin(cross_class_dupes.index)][["class_name", "file_path", "sha256"]])


Duplicate groups spanning more than one class: 172
                            class_name  \
53703    Staple Gun Dataset Collection   
36043      Nail Gun Dataset Collection   
35914      Nail Gun Dataset Collection   
35910      Nail Gun Dataset Collection   
35959      Nail Gun Dataset Collection   
...                                ...   
65872               Wirecutter Dataset   
56949  Tape Measure Dataset Collection   
55938             Tape Measure Dataset   
35901      Nail Gun Dataset Collection   
53561    Staple Gun Dataset Collection   

                                               file_path  \
53703  /content/dataset_raw/Staple Gun Dataset Collec...   
36043  /content/dataset_raw/Nail Gun Dataset Collecti...   
35914  /content/dataset_raw/Nail Gun Dataset Collecti...   
35910  /content/dataset_raw/Nail Gun Dataset Collecti...   
35959  /content/dataset_raw/Nail Gun Dataset Collecti...   
...                                                  ...   
65872  /content/dataset_

## Resolution  Audit.
Flag images whose shorter side falls below a minimum usable threshold -
these get upsampled heavily to reach 224x224 and mostly contribute blur/noise.


In [ ]:
MIN_SHORT_SIDE_PX = 150

In [ ]:
clean_df["short_side"] = clean_df[["width", "height"]].min(axis=1)
clean_df["low_resolution"] = clean_df["short_side"] < MIN_SHORT_SIDE_PX


In [ ]:
low_res_summary = (
    clean_df.groupby("class_name")["low_resolution"]
    .agg(low_res_count="sum", total="count")
)

In [ ]:
low_res_summary["low_res_pct"] = (
    100 * low_res_summary["low_res_count"] / low_res_summary["total"]
).round(2)
print(low_res_summary.sort_values("low_res_pct", ascending=False))


                                      low_res_count  total  low_res_pct
class_name                                                             
Air Compressors Dataset                           0    981          0.0
Angle Grinder Dataset Collection                  0    786          0.0
Axe AI Dataset Collection                         0    643          0.0
Ball Valve Dataset                                0    973          0.0
Bearing Dataset                                   0    969          0.0
...                                             ...    ...          ...
Voltage Tester AI Dataset Collection              0    890          0.0
Welding Gloves Dataset                            0    993          0.0
Wheel Barrow Dataset                              0    930          0.0
Wirecutter Dataset                                0    878          0.0
Wrench AI Dataset Collection                      0    803          0.0

[89 rows x 3 columns]


Per-class summary combining counts, duplicates, and low-resolution rate.


In [ ]:

summary = (
    clean_df.groupby("class_name")
    .agg(
        total_images=("file_path", "count"),
        duplicate_images=("is_duplicate", "sum"),
        low_res_images=("low_resolution", "sum"),
        avg_width=("width", "mean"),
        avg_height=("height", "mean"),
    )
    .round(1)
    .sort_values("total_images")
)

In [ ]:
print(summary)

                                total_images  duplicate_images  \
class_name                                                       
Toolbelt Dataset                          72                 0   
Paint Spray Gun Dataset                  196                 4   
Digital Torque Wrench Dataset            270                23   
Generator Dataset Collection             284                17   
Corner Trowel Dataset                    309                12   
...                                      ...               ...   
Stainless Steel Washer Dataset           989                25   
Metal Nut Dataset                        990                45   
Traffic Cone Dataset                     992                91   
Welding Gloves Dataset                   993                93   
Socket Wrench Dataset                    996                25   

                                low_res_images  avg_width  avg_height  
class_name                                                           

## Export audit_report.csv

In [ ]:
AUDIT_REPORT_PATH = Path("/content/audit_report.csv")
summary.to_csv(AUDIT_REPORT_PATH)
print(f"Saved {AUDIT_REPORT_PATH}")


Saved /content/audit_report.csv


Export manifest.json - one entry per image, with everything needed for the
next ETL stage (filtering + resize) to run without re-touching the raw files.

In [ ]:
manifest_records = clean_df[[
    "file_path",
    "class_name",
    "source_batch",
    "sha256",
    "width",
    "height",
    "is_duplicate",
    "low_resolution",
]].to_dict(orient="records")

In [ ]:

MANIFEST_PATH = Path("/content/manifest.json")

In [ ]:
with open(MANIFEST_PATH, "w") as f:
    json.dump(manifest_records, f, indent=2)


In [ ]:
print(f"Saved {MANIFEST_PATH} with {len(manifest_records)} entries")


Saved /content/manifest.json with 67119 entries


In [ ]:
DRIVE_ARTIFACTS_DIR = Path("/content/drive/MyDrive/hardware-store/audit")

In [ ]:
DRIVE_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
shutil.copy(AUDIT_REPORT_PATH, DRIVE_ARTIFACTS_DIR / AUDIT_REPORT_PATH.name)
shutil.copy(MANIFEST_PATH, DRIVE_ARTIFACTS_DIR / MANIFEST_PATH.name)


PosixPath('/content/drive/MyDrive/hardware-store/audit/manifest.json')

In [ ]:
print(f"Artifacts copied over to {DRIVE_ARTIFACTS_DIR}")

Artifacts copied over to /content/drive/MyDrive/hardware-store/audit


In [ ]:
!cd dataset_raw/ && find . -maxdepth 3 -type f | head -100

In [ ]:
!find . -maxdepth 2 -type d | sort

In [ ]:
!du -h --max-depth=2 | sort -h | tail -30

In [ ]:
!find . -type f | sed 's/.*\.//' | \
!sort | uniq -c | sort -nr

## ETL script for the Hardware-Store tool dataset.

In [ ]:
import json
import shutil
import zipfile
from pathlib import Path

In [ ]:
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm


In [ ]:
# Load the per-image manifest produced by the audit stage.

In [ ]:
MANIFEST_PATH = Path("/content/manifest.json")

In [ ]:
with open(MANIFEST_PATH) as f:
    manifest_records = json.load(f)

In [ ]:
df = pd.DataFrame(manifest_records)

In [ ]:
print(f"Manifest entries: {len(df)}")
print(f"Classes in manifest: {df['class_name'].nunique()}")


Manifest entries: 67119
Classes in manifest: 89


In [ ]:
# Filter 1: drop the Toolbelt class (only 72 images, too few to be viable).

In [ ]:

CLASSES_TO_DROP = ["Toolbelt Dataset"]

In [ ]:
before = len(df)
df = df[~df["class_name"].isin(CLASSES_TO_DROP)].copy()
print(f"Dropped {before - len(df)} images from {CLASSES_TO_DROP}")
print(f"Classes remaining: {df['class_name'].nunique()}")


Dropped 72 images from ['Toolbelt Dataset']
Classes remaining: 88


## Filter 2:
drop exact duplicates by SHA-256, keeping the first occurrence.
For cross-class duplicates (same hash in two classes), keeping "first" assigns
the image to whichever class appears first in the manifest order. These were
reported in the audit; with exact-hash dedup this is the accepted resolution.

In [ ]:
before = len(df)
df = df.drop_duplicates(subset="sha256", keep="first").copy()
print(f"Removed {before - len(df)} exact duplicate images")
print(f"Unique images remaining: {len(df)}")


Removed 2657 exact duplicate images
Unique images remaining: 64390


Normalize class names into clean label slugs for folder names.

Strips the trailing "Dataset" / "AI Dataset" / "Collection" noise so the
training folders read as clean class labels (used later as model class names).

In [ ]:
def clean_label(raw_name: str) -> str:
    """Turn a raw category folder name into a clean lowercase underscore slug."""
    name = raw_name

    for suffix in ["Dataset Collection", "AI Dataset Collection",
                   "AI Dataset", "Dataset","Dataset Collection ","Dataset ",
                   "AI","AI Dataset Collection","AI Dataset Collection" ]:

        if name.endswith(suffix):

            name = name[: -len(suffix)]
            break

    return "_".join(name.strip().lower().split())

In [ ]:
df["label"] = df["class_name"].apply(clean_label)
print(df[["class_name", "label"]].drop_duplicates().to_string(index=False))


                          class_name                        label
             Air Compressors Dataset              air_compressors
    Angle Grinder Dataset Collection                angle_grinder
           Axe AI Dataset Collection                       axe_ai
                  Ball Valve Dataset                   ball_valve
                     Bearing Dataset                      bearing
     Caulking Gun Dataset Collection                 caulking_gun
                      Chisel Dataset                       chisel
            Clamp Dataset Collection                        clamp
   Concrete Mixer Dataset Collection               concrete_mixer
       Construction Earmuffs Dataset        construction_earmuffs
               Corner Trowel Dataset                corner_trowel
          Crowbar Dataset Collection                      crowbar
  Digital Caliper Dataset Collection              digital_caliper
       Digital Torque Wrench Dataset        digital_torque_wrench
     Drill

##  Stratified train/val split.
Split within each class so every class keeps the same train/val ratio,
regardless of how imbalanced the class sizes are.

In [ ]:
VAL_FRACTION = 0.15
RANDOM_SEED = 42

In [ ]:
train_parts = []
val_parts = []

In [ ]:
for label, group in df.groupby("label"):
    shuffled = group.sample(frac=1.0, random_state=RANDOM_SEED)
    n_val = max(1, int(round(len(shuffled) * VAL_FRACTION)))
    val_parts.append(shuffled.iloc[:n_val])
    train_parts.append(shuffled.iloc[n_val:])

In [ ]:
train_df = pd.concat(train_parts).reset_index(drop=True)
val_df = pd.concat(val_parts).reset_index(drop=True)


In [ ]:
train_df["split"] = "train"
val_df["split"] = "val"

In [ ]:
split_df = pd.concat([train_df, val_df]).reset_index(drop=True)

In [ ]:
print(f"Train images: {len(train_df)}")
print(f"Val images:   {len(val_df)}")
print(f"Total:        {len(split_df)}")


Train images: 54735
Val images:   9655
Total:        64390


Resize transform: resize shorter side to 224, then center crop to 224x224.

This is the standard ImageNet inference pipeline.

Aspect ratio is preservedby the resize;
the center crop trims the longer dimension symmetrically.

In [ ]:
TARGET_SIZE = 224

In [ ]:
def resize_and_center_crop(img: Image.Image, size: int = TARGET_SIZE) -> Image.Image:
    """Resize the short side to `size`, then center crop to size x size."""
    width, height = img.size
    scale = size / min(width, height)
    new_width = round(width * scale)
    new_height = round(height * scale)
    img = img.resize((new_width, new_height), Image.BILINEAR)

    left = (new_width - size) // 2
    top = (new_height - size) // 2

    return img.crop((left, top, left + size, top + size))

Process every image: open, convert to RGB, transform, write as JPEG.
Output layout: <OUTPUT_ROOT>/<split>/<label>/<sha256>.jpg

Using the sha256 as filename guarantees no name collisions across source batches.

In [ ]:
OUTPUT_ROOT = Path("/content/hardware_store_224")


In [ ]:
if OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True)


In [ ]:
JPEG_QUALITY = 90
processed_records = []
failed = []

In [ ]:

for row in tqdm(split_df.itertuples(index=False), total=len(split_df),
                desc="Resizing images"):
    out_dir = OUTPUT_ROOT / row.split / row.label
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{row.sha256}.jpg"

    try:
        with Image.open(row.file_path) as img:
            img = img.convert("RGB")
            img = resize_and_center_crop(img, TARGET_SIZE)
            img.save(out_path, "JPEG", quality=JPEG_QUALITY)

        processed_records.append({
            "sha256": row.sha256,
            "label": row.label,
            "split": row.split,
            "relative_path": str(out_path.relative_to(OUTPUT_ROOT)),
        })
    except Exception as exc:
        failed.append({"file_path": row.file_path, "error": str(exc)})


Resizing images:   0%|          | 0/64390 [00:00<?, ?it/s]

In [ ]:
print(f"Processed: {len(processed_records)}")
print(f"Failed:    {len(failed)}")
if failed:
    print(pd.DataFrame(failed).head(10).to_string(index=False))


Processed: 64390
Failed:    0


## Final class-count sanity check on the processed set.

In [ ]:
processed_df = pd.DataFrame(processed_records)
counts = (
    processed_df.groupby(["label", "split"])
    .size()
    .unstack(fill_value=0)
    .sort_values("train")
)

In [ ]:
counts["total"] = counts.sum(axis=1)
print(counts.to_string())
print()
print(f"Final class count: {processed_df['label'].nunique()}")
print(f"Final image count: {len(processed_df)}")


split                         train  val  total
label                                          
paint_spray_gun                 165   29    194
digital_torque_wrench           218   38    256
generator                       234   41    275
staple_gun                      248   44    292
corner_trowel                   258   45    303
float_tool                      261   46    307
construction_earmuffs           273   48    321
paint_respirator                303   54    357
infrared_digital_thermometer    307   54    361
hoe_tool                        351   62    413
respirator_mask_n95             359   63    422
masking_tape_roll               380   67    447
concrete_mixer                  384   68    452
paint_tray                      403   71    474
paint_roller_ai                 416   74    490
safety_glasses                  432   76    508
crowbar                         434   77    511
jack_hammer                     451   80    531
clamp                           467   82

Write the processed manifest - the record of exactly what ended up in the
packaged dataset (label, split, path inside the zip).

In [ ]:
PROCESSED_MANIFEST_PATH = OUTPUT_ROOT / "processed_manifest.json"

In [ ]:
with open(PROCESSED_MANIFEST_PATH, "w") as f:
    json.dump(processed_records, f, indent=2)

In [ ]:

class_list = sorted(processed_df["label"].unique().tolist())

In [ ]:
CLASS_LIST_PATH = OUTPUT_ROOT / "classes.json"

In [ ]:
with open(CLASS_LIST_PATH, "w") as f:
    json.dump(class_list, f, indent=2)

In [ ]:
print(f"Saved {PROCESSED_MANIFEST_PATH}")
print(f"Saved {CLASS_LIST_PATH} ({len(class_list)} classes)")

Saved /content/hardware_store_224/processed_manifest.json
Saved /content/hardware_store_224/classes.json (87 classes)


Package the processed dataset into a single zip.

One zip is far faster to move to Drive and to unzip in the training runtime
than thousands of individual small files over a mounted Drive.

In [ ]:
ZIP_PATH = Path("/content/hardware_store_224.zip")
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

In [ ]:
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_STORED) as zf:
    for file_path in tqdm(list(OUTPUT_ROOT.rglob("*")), desc="Zipping"):
        if file_path.is_file():
            zf.write(file_path, file_path.relative_to(OUTPUT_ROOT))


Zipping:   0%|          | 0/64568 [00:00<?, ?it/s]

In [ ]:
zip_size_mb = ZIP_PATH.stat().st_size / (1024 * 1024)
print(f"Created {ZIP_PATH} ({zip_size_mb:.1f} MB)")

Created /content/hardware_store_224.zip (881.1 MB)


Copy artifacts to Drive

In [ ]:
DRIVE_DIR = Path("/content/drive/MyDrive/hardware-store/ETL")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
shutil.copy(ZIP_PATH, DRIVE_DIR / ZIP_PATH.name)
print(f"Copied {ZIP_PATH.name} to {DRIVE_DIR}")

Copied hardware_store_224.zip to /content/drive/MyDrive/hardware-store/ETL
